# 📊 Ingesta de Datos Freddie Mac - Capa Bronze

## Objetivo
Este notebook realiza la **ingesta inicial** de datos de préstamos hipotecarios de Freddie Mac desde archivos `.txt` almacenados en un **UC Volume**.

## Arquitectura Medallion
- **Bronze (Raw)**: Ingesta de datos sin transformaciones (este notebook)
- **Silver**: Limpieza y normalización de datos
- **Gold**: Datos agregados listos para análisis

## Fuente de Datos
**Freddie Mac Single-Family Loan-Level Dataset**
- Contiene información de préstamos hipotecarios
- Tipos de archivo:
  - `sample_orig_*.txt`: Datos de **originación** del préstamo (características iniciales)
- Formato: Archivos de texto delimitados por pipe (`|`)
- Sin encabezados (headers)

---
## 📝 Paso 1: Definir el Schema de Originación

### ¿Por qué definir el schema?
Los archivos de Freddie Mac **no tienen encabezados** (headers), por lo que necesitamos:
1. Definir explícitamente los nombres de las columnas
2. Especificar los tipos de datos

### Campos Principales del Schema de Originación:
- **credit_score**: Puntuación crediticia del prestatario
- **loan_id**: Identificador único del préstamo
- **original_upb**: Saldo principal original (Unpaid Principal Balance)
- **original_interest_rate**: Tasa de interés original
- **property_state**: Estado donde se ubica la propiedad
- **loan_purpose**: Propósito del préstamo (compra, refinanciación, etc.)

💡 **Nota**: Por ahora usamos `StringType` para todos los campos. En la capa Silver haremos las conversiones a tipos numéricos y fechas.

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType
)

# Schema completo de originación de Freddie Mac (27 campos)
# Todos los campos como String para ingesta raw - convertiremos tipos en Silver
ORIGINATION_SCHEMA = StructType([
    StructField("credit_score",                         StringType(), True),
    StructField("first_payment_date",                   StringType(), True),
    StructField("first_time_homebuyer_flag",            StringType(), True),
    StructField("maturity_date",                        StringType(), True),
    StructField("msa",                                  StringType(), True),
    StructField("mip",                                  StringType(), True),
    StructField("number_of_units",                      StringType(), True),
    StructField("occupancy_status",                     StringType(), True),
    StructField("original_cltv",                        StringType(), True),
    StructField("original_dti",                         StringType(), True),
    StructField("original_upb",                         StringType(), True),
    StructField("original_ltv",                         StringType(), True),
    StructField("original_interest_rate",               StringType(), True),
    StructField("channel",                              StringType(), True),
    StructField("ppm_flag",                              StringType(), True),
    StructField("product_type",                         StringType(), True),
    StructField("property_state",                       StringType(), True),
    StructField("property_type",                        StringType(), True),
    StructField("zip_code",                              StringType(), True),
    StructField("loan_id",                               StringType(), True),
    StructField("loan_purpose",                         StringType(), True),
    StructField("original_loan_term",                   StringType(), True),
    StructField("number_of_borrowers",                  StringType(), True),
    StructField("seller_name",                           StringType(), True),
    StructField("servicer_name",                        StringType(), True),
    StructField("super_conforming_flag",                StringType(), True),
    StructField("pre_harp_loan_id",                     StringType(), True),
])

---
## 💾 Paso 2: Leer Archivos desde UC Volume

### Unity Catalog Volumes
Los **UC Volumes** son el sistema de almacenamiento de archivos en Unity Catalog que permite:
- Gestionar archivos no estructurados (txt, csv, json, parquet, etc.)
- Control de acceso granular
- Organización en 3 niveles: `catalog.schema.volume`

### Path del Volume:
```
/Volumes/credit_risk_platform/bronze/raw_files
```

### Características del Formato:
- **Delimitador**: Pipe (`|`) en lugar de coma
- **Sin encabezados**: `header=false`
- **Valores nulos**: Representados como strings vacíos

### Metadata de Archivos
Usamos la columna especial `_metadata.file_path` para rastrear de qué archivo proviene cada registro. Esto es útil para:
- Debugging
- Auditoría
- Identificar el año de los datos (los nombres tienen el año: `sample_orig_2006.txt`)

In [0]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
volume_path = "/Volumes/credit_risk_platform/bronze/raw_files"

# ============================================================
# LECTURA DE ARCHIVOS
# ============================================================
# Leer todos los archivos .txt del volumen usando el schema definido
# Delimitador: pipe (|), sin encabezados, strings vacíos = NULL
df = spark.read.format("csv") \
    .option("sep", "|") \
    .option("header", "false") \
    .option("nullValue", "") \
    .schema(ORIGINATION_SCHEMA) \
    .load(f"{volume_path}/sample_orig_*.txt")

# ============================================================
# AGREGAR METADATA
# ============================================================
# Añadir columna con el path completo del archivo fuente
# Útil para auditoría y para extraer el año del nombre del archivo
df = df.withColumn("_source_file", df["_metadata"].file_path)

# ============================================================
# VALIDACIÓN Y PREVIEW
# ============================================================
print(f"✅ Total de filas leídas: {df.count():,}")
print(f"\n🔍 Primeras 10 filas:")
display(df.limit(10))

---
## 💾 Paso 3: Guardar en Tabla Delta (Capa Bronze)

### ¿Por qué Delta Lake?
Delta Lake es el formato de almacenamiento estándar en Databricks que proporciona:
- **ACID Transactions**: Garantiza consistencia de datos
- **Time Travel**: Permite consultar versiones anteriores de los datos
- **Schema Evolution**: Facilita cambios de schema sin romper pipelines
- **Mejor performance**: Optimizaciones automáticas de lectura/escritura

### Estrategia de Ingesta
- **Modo**: `append` - Agregar nuevos datos sin sobrescribir existentes
- **Timestamp de ingesta**: Columna `_ingestion_timestamp` para auditoría
- **Tabla destino**: `credit_risk_platform.bronze.origination`

### Metadata Agregada:
- `_source_file`: Path del archivo fuente (ya agregado en paso anterior)
- `_ingestion_timestamp`: Momento exacto de la ingesta

💡 **Nota**: En futuras ejecuciones, si queremos evitar duplicados, deberíamos implementar lógica de deduplicación o usar `merge` en lugar de `append`.

In [0]:
from pyspark.sql.functions import current_timestamp

# ============================================================
# AGREGAR TIMESTAMP DE INGESTA
# ============================================================
# Añadir columna con la fecha/hora exacta de la ingesta
# Útil para tracking, auditoría y troubleshooting
df_final = df.withColumn("_ingestion_timestamp", current_timestamp())

# ============================================================
# GUARDAR EN TABLA DELTA - CAPA BRONZE
# ============================================================
# Tabla destino: credit_risk_platform.bronze.origination
# Modo: append (agregar datos sin sobrescribir)
# Formato: Delta Lake (ACID, Time Travel, optimizaciones)
table_name = "credit_risk_platform.bronze.origination"

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print(f"✅ Datos guardados exitosamente en: {table_name}")
print(f"📊 Total de registros escritos: {df_final.count():,}")

---
## ✔️ Paso 4: Validar Tabla Bronze

Verificar que los datos se guardaron correctamente en la tabla Delta.

In [0]:
# ============================================================
# LEER Y VALIDAR TABLA BRONZE
# ============================================================
bronze_df = spark.table("credit_risk_platform.bronze.origination")

print("="*60)
print("📊 RESUMEN DE TABLA BRONZE")
print("="*60)

# 1. Conteo total de registros en la tabla
total_records = bronze_df.count()
print(f"\n1️⃣ Total de registros en tabla: {total_records:,}")

# 2. Verificar las columnas
print(f"\n2️⃣ Columnas en la tabla: {len(bronze_df.columns)}")
print(f"   Columnas de negocio: {len(ORIGINATION_SCHEMA.fields)}")
print(f"   Columnas de metadata: _source_file, _ingestion_timestamp")

# 3. Ver distribución por archivo fuente
print(f"\n3️⃣ Distribución por archivo fuente:")
bronze_df.groupBy("_source_file").count() \
    .orderBy("_source_file") \
    .show(truncate=False)

# 4. Ver últimas ingestas
print(f"\n4️⃣ Últimas 5 ingestas (por timestamp):")
bronze_df.select("_ingestion_timestamp", "loan_id", "credit_score", "original_upb", "_source_file") \
    .orderBy("_ingestion_timestamp", ascending=False) \
    .limit(5) \
    .show(truncate=50)

print("\n" + "="*60)
print("✅ Validación completada")
print("="*60)